# 🧪 Step-by-Step Node Testing - Chrono S. Thompson

This notebook executes and tests each **Node** in the LangGraph pipeline individually and sequentially, inspecting `ChronoState` mutations at each step.

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Ensure project root is set and present in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")

from src.chrono_s_thompson.core.state import ChronoState
from src.chrono_s_thompson.graph.nodes import (
    fetch_events_node,
    batch_events_node,
    rank_events_node,
    take_photograph_node,
    index_selected_event,
    search_context_node,
    rerank_context_node,
    write_article_node,
    verify_article_node,
    publish_article_node,
)

print("✅ Modules and Nodes imported successfully!")
print(f"Working Directory: {os.getcwd()}")

## 0. State Initialization (`ChronoState`)

In [ ]:
# Define target date in MM/DD format
state = ChronoState(target_date="05/20")
print(f"Target Date: {state.target_date}")
print(f"Initial State: {state.model_dump()}")

## 1. Node `fetch_events`
Fetches historical events for the target date via the FastMCP Server (`stdio`).

In [ ]:
update = await fetch_events_node(state)
state = state.model_copy(update=update)

if state.error:
    print(f"❌ Error in fetch_events: {state.error_msg}")
else:
    print(f"Total events retrieved via MCP: {len(state.raw_events)}")
    for event in state.raw_events[:3]:
        print(f" - [{event.year}] {event.title} ({event.page_name})")

## 2. Node `batch_events`
Filters and structures raw historical events payload using LLM structured output.

In [ ]:
update = await batch_events_node(state)
state = state.model_copy(update=update)

if state.error:
    print(f"❌ Error in batch_events: {state.error_msg}")
else:
    print(f"Total filtered/batched events: {len(state.batched_events.events)}")
    for e in state.batched_events.events[:3]:
        print(f" - [{e.year}] {e.title}")

## 3. Node `rank_events`
Selects the event with the highest narrative tension and formulates a Gonzo editorial hook.

In [ ]:
update = await rank_events_node(state)
state = state.model_copy(update=update)

if state.curated_story:
    print(f"Selected Event: [{state.curated_story.selected_event.year}] {state.curated_story.selected_event.title}")
    print(f"Gonzo Hook: {state.curated_story.gonzo_hook}")
    print(f"Photo Description: {state.curated_story.photo_description}")
    print(f"RAG Query String: {state.curated_story.query_string}")

## 4. Node `photographer`
Synthesizes a visual historical photograph illustration using image generation.

In [ ]:
update = await take_photograph_node(state)
state = state.model_copy(update=update)

print(f"Saved Photo Filename: {state.photo_filename}")
print(f"Full Photo File Path: {state.photo_file_path}")

## 5. Node `index_selected_event`
Scrapes Wikipedia content for the selected event, cleans wikitext clutter, and creates the vector store retriever.

In [ ]:
update = await index_selected_event(state)
state = state.model_copy(update=update)

if state.detailed_event:
    print(f"Retrieved Details Title: {state.detailed_event.title}")
    print(f"Source URL: {state.detailed_event.url}")
print(f"Vector Store Retriever Created: {state.retriever is not None}")

## 6. Node `search_context`
Queries vector database for text chunks relevant to the story and hook.

In [ ]:
update = await search_context_node(state)
state = state.model_copy(update=update)

print(f"Retrieved Documents from Vector Search: {len(state.retrieved_docs)}")

## 7. Node `rerank_context` (Conditional Routing)

> **Edge Condition Notice:** As defined in `builder.py` (`route_after_search_context`), if `len(state.retrieved_docs) < 1`, the graph branches directly to `write_article`. Otherwise, it executes `rerank_context` to deduplicate, score by relevance, and assign stable source IDs (`S1`, `S2`...).

In [ ]:
if len(state.retrieved_docs) < 1:
    print("ℹ️ No documents retrieved. Bypassing rerank_context (routing directly to write_article).")
else:
    update = await rerank_context_node(state)
    state = state.model_copy(update=update)
    print(f"Re-ranked Documents Count: {len(state.reranked_docs)}")
    print(f"Mapped Sources: {list(state.sources.keys())}")
    for s_id, meta in state.sources.items():
        print(f"  [{s_id}] {meta.title} -> {meta.url}")

## 8. Node `write_article`
Drafts high-engagement prose in Gonzo style with inline citation tags (`[S1]`, `[S2]`) and collapsible sources section.

In [ ]:
update = await write_article_node(state)
state = state.model_copy(update=update)

print("=== ARTICLE DRAFT PREVIEW ===")
print(state.draft_article[:500] if state.draft_article else "No draft generated.")

## 9. Node `verify_article`
Performs citation regex checks and structured factual claim verification via LLM verifier node.

In [ ]:
update = await verify_article_node(state)
state = state.model_copy(update=update)

if state.verification_result:
    print(f"Article Valid? {state.verification_result.is_valid}")
    print(f"Cited IDs Found: {state.verification_result.cited_ids}")
    print(f"Missing Citation IDs: {state.verification_result.missing_citation_ids}")
    print(f"Revision Feedback: {state.verification_feedback}")

## 10. Node `publish_article`
Persiste o despacho verificado em Markdown no diretório `storage/output/`.

In [ ]:
update = await publish_article_node(state)
state = state.model_copy(update=update)

print(f"Published Dispatch Path: {state.published_path}")
print("✅ Success! Pipeline tested step-by-step end-to-end.")